# LeetCode #117 — Populating Next Right Pointers in Each Node II: cây bất kỳ, dummy cho tầng dưới

**Mức độ:** Medium · **Chủ đề:** Linked List, Tree, Depth-First Search, Breadth-First Search, Binary Tree

Skeleton trên LeetCode:

```python
class Solution:
    def connect(self, root: 'Node') -> 'Node':
```

Khác #116: cây **không** còn hoàn hảo — nút có thể thiếu con, các tầng có "lỗ". Hai quy tắc nối của #116 không dùng được nữa.

Mục tiêu của notebook:

1. Cách 1: BFS theo tầng — vẫn chạy y nguyên (không phụ thuộc hình dạng cây).
2. Cách 2 (`O(1)` bộ nhớ): đi ngang tầng trên bằng `next`, **xây tầng dưới như một danh sách liên kết** với **nút giả (dummy)** và con trỏ `tail`.
3. Hiểu vì sao đệ quy kiểu #116 thất bại ở đây.

# 1. Bài toán là gì?

Cho cây nhị phân **bất kỳ**, mỗi nút có thêm con trỏ `next`. Gán `next` của mỗi nút trỏ tới nút **bên phải gần nhất cùng tầng**; không có thì `None`.

**Constraints (theo đề):**

- Số nút trong khoảng `[0, 6000]`.
- `-100 <= Node.val <= 100`

**Follow up (theo đề):** chỉ dùng bộ nhớ phụ hằng số (ngăn xếp đệ quy không tính).

---

## Ví dụ

```text
        1                  1 -> None
       / \                / \
      2   3       ->     2 -> 3 -> None
     / \   \            / \    \
    4   5   7          4 -> 5 -> 7 -> None

LeetCode in:  [1, #, 2, 3, #, 4, 5, 7, #]

root = []  ->  []
```

Chú ý `5.next = 7`: hai nút khác cha, và `3` không có con trái — phải "nhảy qua" chỗ trống.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Cây nhị phân (binary tree)
2. Duyệt theo chiều rộng (BFS — Breadth-First Search)
3. Danh sách liên kết đơn (singly linked list)

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Cây nhị phân (binary tree)

### Cây nhị phân là gì?

Cây nhị phân gồm các **nút**; mỗi nút có một giá trị và tối đa **hai con**: con **trái** và con **phải**. Nút trên cùng là **gốc** (root); nút không có con là **lá** (leaf).

```text
          1          ← gốc (độ sâu 0)
        /   \
       2     3       ← độ sâu 1
      / \     \
     4   5     6     ← lá 4, 5, 6 (độ sâu 2)
```

Thuật ngữ: **cha/con**, **anh em** (cùng cha), **cây con** (một nút cùng toàn bộ con cháu), **độ sâu** (số cạnh từ gốc), **chiều cao** (số tầng / đường dài nhất từ gốc tới lá).

In [1]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Dựng cây ở hình trên bằng tay
goc = TreeNode(1,
               TreeNode(2, TreeNode(4), TreeNode(5)),
               TreeNode(3, None, TreeNode(6)))
print("gốc:", goc.val, "| con trái:", goc.left.val, "| con phải:", goc.right.val,
      "| 3 có con trái?", goc.right.left)

gốc: 1 | con trái: 2 | con phải: 3 | 3 có con trái? None


### Biểu diễn kiểu LeetCode: danh sách theo tầng (level-order)

LeetCode ghi cây thành danh sách đọc từng tầng, từ trái sang phải, `None` cho chỗ trống: cây trên là `[1, 2, 3, 4, 5, None, 6]`. Dựng cây từ danh sách này bằng một **hàng đợi**: lần lượt lấy nút cha ra, gán hai giá trị kế tiếp làm con trái/phải.

In [2]:
from collections import deque

def tao_cay(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    q, i = deque([root]), 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def ve_cay(node, pre="", la_phai=True, goc_cay=True):
    """In cây nằm ngang để dễ nhìn."""
    if node is None:
        return
    ve_cay(node.right, pre + ("" if goc_cay else ("    " if la_phai else "│   ")), True, False)
    print(pre + ("" if goc_cay else ("┌── " if la_phai else "└── ")) + str(node.val))
    ve_cay(node.left, pre + ("" if goc_cay else ("│   " if la_phai else "    ")), False, False)

ve_cay(tao_cay([1, 2, 3, 4, 5, None, 6]))

    ┌── 6
┌── 3
1
│   ┌── 5
└── 2
    └── 4


(Cây in nằm ngang: gốc ở bên trái, con **phải** ở phía **trên**, con **trái** ở phía **dưới**.)

### Ba cách duyệt theo chiều sâu

Với mỗi nút ta làm ba việc: thăm **nút**, duyệt cây con **trái**, duyệt cây con **phải**. Thứ tự của ba việc cho ba kiểu duyệt:

| Kiểu | Thứ tự | Cây trên |
|---|---|---|
| **Preorder** (tiền thứ tự) | nút → trái → phải | `1 2 4 5 3 6` |
| **Inorder** (trung thứ tự) | trái → nút → phải | `4 2 5 1 3 6` |
| **Postorder** (hậu thứ tự) | trái → phải → nút | `4 5 2 6 3 1` |

Mẹo nhớ: "pre/in/post" chỉ **vị trí của nút** so với hai cây con.

In [3]:
def preorder(node, out):
    if node is None:
        return
    out.append(node.val)        # nút
    preorder(node.left, out)    # trái
    preorder(node.right, out)   # phải

def inorder(node, out):
    if node is None:
        return
    inorder(node.left, out)
    out.append(node.val)
    inorder(node.right, out)

def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)

cay = tao_cay([1, 2, 3, 4, 5, None, 6])
for ten, f in [("preorder", preorder), ("inorder", inorder), ("postorder", postorder)]:
    out = []; f(cay, out); print(f"{ten:<9}:", out)

preorder : [1, 2, 4, 5, 3, 6]
inorder  : [4, 2, 5, 1, 3, 6]
postorder: [4, 5, 2, 6, 3, 1]


### Duyệt không đệ quy bằng ngăn xếp

Đệ quy thực chất dùng *call stack*; ta có thể tự quản lý ngăn xếp. Preorder: lấy nút ra, ghi lại, đẩy con **phải trước** rồi con **trái** (để trái được lấy ra trước). Inorder: đi trái hết mức, lấy ra, rồi sang phải.

In [4]:
def preorder_lap(root):
    out, st = [], [root] if root else []
    while st:
        node = st.pop()
        out.append(node.val)
        if node.right: st.append(node.right)
        if node.left:  st.append(node.left)
    return out

def inorder_lap(root):
    out, st, cur = [], [], root
    while cur or st:
        while cur:              # đi trái hết mức
            st.append(cur); cur = cur.left
        cur = st.pop()
        out.append(cur.val)
        cur = cur.right
    return out

print("preorder lặp:", preorder_lap(cay), "| inorder lặp:", inorder_lap(cay))

preorder lặp: [1, 2, 4, 5, 3, 6] | inorder lặp: [4, 2, 5, 1, 3, 6]


### Tư duy đệ quy trên cây

Hầu hết bài về cây giải bằng cách hỏi: "nếu đã biết đáp án cho **cây con trái** và **cây con phải**, ghép lại thế nào?". Ví dụ chiều cao, số nút, tổng giá trị:

In [5]:
def chieu_cao(node):
    if node is None:
        return 0                                   # cây rỗng cao 0
    return 1 + max(chieu_cao(node.left), chieu_cao(node.right))

def so_nut(node):
    return 0 if node is None else 1 + so_nut(node.left) + so_nut(node.right)

print("chiều cao:", chieu_cao(cay), "| số nút:", so_nut(cay))

chiều cao: 3 | số nút: 6


### Lỗi hay gặp

- Quên trường hợp `node is None` → `AttributeError: 'NoneType' object has no attribute 'left'`.
- Nhầm "lá" (không có con nào) với "nút thiếu một con".
- Nhầm thứ tự preorder/inorder/postorder.
- Cây lệch (như danh sách) cao tới `n` → đệ quy sâu `n` tầng.

## 2.2 Duyệt theo chiều rộng (BFS — Breadth-First Search)

### BFS là gì?

BFS duyệt đồ thị/cây **theo từng lớp**: thăm điểm xuất phát, rồi **mọi** điểm cách 1 bước, rồi mọi điểm cách 2 bước, … giống vết sóng lan ra khi ném hòn đá xuống nước.

Công cụ: **hàng đợi** (FIFO). Điểm phát hiện trước được xử lý trước → các điểm gần luôn được xử lý trước các điểm xa.

```text
        A            lớp 0: A
       / \           lớp 1: B, C
      B   C          lớp 2: D, E, F
     / \   \
    D   E   F        thứ tự BFS: A B C D E F
```

### Khung BFS trên đồ thị

```text
q = deque([start]); visited = {start}
while q:
    u = q.popleft()
    xử lý u
    for v in hàng_xóm(u):
        if v not in visited:
            visited.add(v)        # đánh dấu NGAY khi đưa vào hàng đợi
            q.append(v)
```

In [6]:
from collections import deque

do_thi = {                       # danh sách kề
    "A": ["B", "C"], "B": ["A", "D", "E"], "C": ["A", "F"],
    "D": ["B"], "E": ["B", "F"], "F": ["C", "E"],
}

def bfs(graph, start):
    q, visited, thu_tu = deque([start]), {start}, []
    while q:
        u = q.popleft()
        thu_tu.append(u)
        for v in graph[u]:
            if v not in visited:
                visited.add(v)
                q.append(v)
        print(f"lấy {u} | hàng đợi: {list(q)}")
    return thu_tu

print("Thứ tự BFS:", bfs(do_thi, "A"))

lấy A | hàng đợi: ['B', 'C']
lấy B | hàng đợi: ['C', 'D', 'E']
lấy C | hàng đợi: ['D', 'E', 'F']
lấy D | hàng đợi: ['E', 'F']
lấy E | hàng đợi: ['F']
lấy F | hàng đợi: []
Thứ tự BFS: ['A', 'B', 'C', 'D', 'E', 'F']


### BFS theo tầng (level-order) trên cây nhị phân

Muốn biết nút nào thuộc tầng nào: ở đầu mỗi vòng, hàng đợi chứa **đúng** các nút của một tầng. Ghi lại `len(q)`, lấy ra đúng chừng ấy nút (và đẩy con của chúng vào) — đó là một tầng.

In [7]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

cay_bfs = TreeNode(3, TreeNode(9), TreeNode(20, TreeNode(15), TreeNode(7)))

def theo_tang(root):
    if not root:
        return []
    q, tang = deque([root]), []
    while q:
        cur = []
        for _ in range(len(q)):          # đúng số nút của tầng hiện tại
            node = q.popleft()
            cur.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        tang.append(cur)
    return tang

print("các tầng:", theo_tang(cay_bfs))

các tầng: [[3], [9, 20], [15, 7]]


### BFS tìm đường ngắn nhất trên lưới

Trên đồ thị **không trọng số** (mỗi bước dài như nhau), lần đầu BFS chạm tới một ô chính là đi bằng **số bước ít nhất**. Lưu `dist` khi đưa ô vào hàng đợi.

In [8]:
luoi = ["S.#.",
        "..#.",
        "...E"]

def ngan_nhat(grid):
    R, C = len(grid), len(grid[0])
    start = next((r, c) for r in range(R) for c in range(C) if grid[r][c] == "S")
    dist = {start: 0}
    q = deque([start])
    while q:
        r, c = q.popleft()
        if grid[r][c] == "E":
            return dist[(r, c)]
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != "#" and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                q.append((nr, nc))
    return -1

print("số bước ít nhất từ S tới E:", ngan_nhat(luoi))

số bước ít nhất từ S tới E: 5


### Độ phức tạp và lỗi hay gặp

`O(V + E)` thời gian (mỗi đỉnh vào hàng đợi một lần, mỗi cạnh xét một/hai lần), `O(V)` bộ nhớ. Trên cây: `O(n)`; trên lưới `R×C`: `O(R·C)`.

- Đánh dấu `visited` khi **lấy ra** thay vì khi **đưa vào** → một ô có thể vào hàng đợi nhiều lần (chậm, đếm sai).
- Dùng `list.pop(0)` → mỗi lần `O(n)`.
- Dùng `pop()` (cuối) → thành DFS, mất tính "gần trước".
- Quên kiểm tra biên lưới.

## 2.3 Danh sách liên kết đơn (singly linked list)

### Danh sách liên kết là gì?

Mảng (`list` của Python) lưu các phần tử **liền nhau** trong bộ nhớ, nên truy cập `a[i]` tức thì. Danh sách liên kết thì khác: mỗi phần tử là một **nút** (node) riêng lẻ, chứa **giá trị** và **con trỏ** `next` chỉ tới nút kế tiếp. Nút cuối có `next = None`.

```text
head
 │
 ▼
┌───┬───┐   ┌───┬───┐   ┌───┬────┐
│ 1 │ ●─┼──►│ 2 │ ●─┼──►│ 3 │None│
└───┴───┘   └───┴───┘   └───┴────┘
 val next
```

Ta chỉ giữ con trỏ tới nút đầu (`head`). Muốn tới nút thứ `k` phải đi qua `k` bước — **không** có truy cập ngẫu nhiên.

| | Mảng | Danh sách liên kết |
|---|---|---|
| Lấy phần tử thứ `i` | `O(1)` | `O(i)` |
| Chèn/xóa ở đầu | `O(n)` | `O(1)` |
| Chèn/xóa sau một nút đã có trong tay | `O(n)` | `O(1)` |

In [9]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

# Tạo bằng tay: 1 -> 2 -> 3
c = ListNode(3)
b = ListNode(2, c)
a = ListNode(1, b)
head = a
print(head.val, head.next.val, head.next.next.val, head.next.next.next)

1 2 3 None


### Duyệt danh sách và hai hàm tiện ích

Mẫu duyệt cơ bản: con trỏ `cur` bắt đầu ở `head`, mỗi bước `cur = cur.next`, dừng khi `cur is None`.

In [10]:
def tao_danh_sach(values):
    """[1, 2, 3] -> 1 -> 2 -> 3, trả về head."""
    dummy = tail = ListNode()          # nút giả để khỏi xử lý riêng nút đầu
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def ra_list(head):
    out, cur = [], head
    while cur:                          # duyệt tới khi hết
        out.append(cur.val)
        cur = cur.next
    return out

h = tao_danh_sach([4, 5, 6, 7])
print("ra_list:", ra_list(h))
dem, cur = 0, h
while cur:
    dem += 1
    cur = cur.next
print("độ dài:", dem)

ra_list: [4, 5, 6, 7]
độ dài: 4


### Chèn và xóa: chỉ cần đổi con trỏ

Chèn nút `x` **sau** nút `p`: `x.next = p.next; p.next = x` (thứ tự quan trọng — làm ngược sẽ mất phần sau).

Xóa nút **sau** `p`: `p.next = p.next.next`.

```text
Chèn x sau p:   p ──► q        ⟹   p ──► x ──► q
Xóa sau p:      p ──► q ──► r  ⟹   p ──────────► r
```

Muốn xóa một nút, ta cần nút **đứng trước** nó. Nếu nút cần xóa là `head` thì không có nút trước → dùng **nút giả (dummy)** đặt trước `head`.

In [11]:
h = tao_danh_sach([1, 2, 4])
p = h.next                              # nút 2
x = ListNode(3)
x.next = p.next                         # 3 -> 4
p.next = x                              # 2 -> 3
print("sau khi chèn 3:", ra_list(h))

def xoa_gia_tri(head, target):
    dummy = ListNode(0, head)           # dummy đứng trước head
    prev = dummy
    while prev.next:
        if prev.next.val == target:
            prev.next = prev.next.next  # bỏ qua nút cần xóa
        else:
            prev = prev.next
    return dummy.next                   # head mới (có thể đã đổi)

print("xóa mọi số 1:", ra_list(xoa_gia_tri(tao_danh_sach([1, 1, 2, 1, 3]), 1)))

sau khi chèn 3: [1, 2, 3, 4]
xóa mọi số 1: [2, 3]


### Đảo ngược danh sách — kỹ thuật ba con trỏ

Đi dọc danh sách, lật mũi tên của từng nút cho trỏ về nút **trước** nó. Cần lưu `nxt` trước khi lật, vì sau khi lật ta mất đường đi tiếp.

```text
prev  cur   nxt
None   1 ──► 2 ──► 3          bước 1: 1.next = None
None ◄─1     2 ──► 3          bước 2: 2.next = 1
None ◄─1 ◄─ 2     3          bước 3: 3.next = 2  → head mới = 3
```

In [12]:
def dao_nguoc(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next      # 1) lưu đường đi tiếp
        cur.next = prev     # 2) lật mũi tên
        prev = cur          # 3) tiến prev
        cur = nxt           # 4) tiến cur
    return prev             # prev là head mới

print("đảo [1..5]:", ra_list(dao_nguoc(tao_danh_sach([1, 2, 3, 4, 5]))))

đảo [1..5]: [5, 4, 3, 2, 1]


### Con trỏ nhanh / chậm (fast & slow)

`slow` đi 1 bước, `fast` đi 2 bước mỗi vòng.

- Khi `fast` tới cuối, `slow` ở **giữa** danh sách.
- Nếu danh sách có **chu trình**, `fast` sẽ đuổi kịp `slow` (thuật toán Floyd).

In [13]:
def nut_giua(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print("giữa [1..5]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5])).val,
      "| giữa [1..6]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5, 6])).val, "(nút giữa thứ hai)")

def co_chu_trinh(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

v = tao_danh_sach([1, 2, 3, 4])
v.next.next.next.next = v.next          # 4 -> 2 tạo chu trình
print("có chu trình?", co_chu_trinh(v), "| danh sách thường:", co_chu_trinh(tao_danh_sach([1, 2])))

giữa [1..5]: 3 | giữa [1..6]: 4 (nút giữa thứ hai)
có chu trình? True | danh sách thường: False


### Lỗi hay gặp với danh sách liên kết

- Truy cập `cur.next.val` khi `cur.next` là `None` → `AttributeError`. Luôn kiểm tra `cur and cur.next`.
- Đổi `cur.next` **trước** khi lưu nút kế → mất phần còn lại của danh sách.
- Quên trường hợp danh sách rỗng (`head is None`) hoặc chỉ một nút.
- Quên dùng dummy khi `head` có thể bị xóa/thay → phải viết nhánh riêng, dễ sai.
- In danh sách có chu trình bằng vòng `while cur` → lặp vô hạn.

## 2.4 Áp dụng vào bài này

Như #116 nhưng cây **bất kỳ**: duyệt tầng hiện tại như một **danh sách liên kết** qua `next`, và dựng tầng dưới bằng **nút giả** + con trỏ `tail` (giống dựng danh sách liên kết) — thay cho hàng đợi của **BFS**.

# 3. Nền tảng

In [14]:
from typing import List, Optional
from collections import deque
import random


class Node:
    def __init__(self, val=0, left=None, right=None, next=None):
        self.val = val
        self.left = left
        self.right = right
        self.next = next


def build_tree(values: list) -> Optional[Node]:
    if not values or values[0] is None:
        return None
    root = Node(values[0])
    q = deque([root])
    i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = Node(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = Node(values[i]); q.append(node.right)
        i += 1
    return root


def serialize_next(root: Optional[Node]) -> list:
    """Theo next từ đầu mỗi tầng. Đầu tầng sau = con đầu tiên tìm được khi đi dọc tầng hiện tại."""
    out, head = [], root
    while head:
        node, nxt_head = head, None
        while node:
            out.append(node.val)
            if nxt_head is None:
                nxt_head = node.left or node.right
            node = node.next
        out.append("#")
        head = nxt_head
    return out


def expected_levels(root):
    """Level order chuẩn (BFS) để đối chiếu."""
    if not root:
        return []
    out, q = [], deque([root])
    while q:
        level = []
        for _ in range(len(q)):
            n = q.popleft()
            level.append(n.val)
            for c in (n.left, n.right):
                if c:
                    q.append(c)
        out += level + ["#"]
    return out

EX = [1, 2, 3, 4, 5, None, 7]
print("Mong đợi:", expected_levels(build_tree(EX)))

Mong đợi:

 [1, '#', 2, 3, '#', 4, 5, 7, '#']


# 4. Vì sao quy tắc của #116 không dùng được?

Trong #116: `cur.right.next = cur.next.left`. Ở đây `cur.next` có thể **không có con nào**, hoặc chỉ có con phải; và nút kế tiếp có con có thể nằm **xa** hơn một bước.

In [15]:
def wrong_116_rules(root):
    def go(node):
        if not node:
            return
        if node.left:
            node.left.next = node.right            # có thể là None dù còn nút khác bên phải
        if node.right and node.next:
            node.right.next = node.next.left       # node.next có thể không có con trái
        go(node.left)
        go(node.right)
    go(root)
    return root

t = build_tree([1, 2, 3, 4, None, None, 5])
print("Mong đợi:", expected_levels(t))
print("Sai     :", serialize_next(wrong_116_rules(build_tree([1, 2, 3, 4, None, None, 5]))))

Mong đợi: [1, '#', 2, 3, '#', 4, 5, '#']
Sai     : [1, '#', 2, 3, '#', 4, '#']


`4` là con trái của `2` (không có con phải), `5` là con phải của `3` — quy tắc của #116 không nối được `4 → 5`.

# 5. Cách 1: BFS theo tầng

Không phụ thuộc hình dạng cây — dùng nguyên như #116.

In [16]:
def connect_bfs(root: Optional[Node]) -> Optional[Node]:
    if not root:
        return root
    q = deque([root])
    while q:
        size = len(q)
        for i in range(size):
            node = q.popleft()
            node.next = q[0] if i < size - 1 else None
            for c in (node.left, node.right):
                if c:
                    q.append(c)
    return root

print(serialize_next(connect_bfs(build_tree(EX))))

[1, '#', 2, 3, '#', 4, 5, 7, '#']


# 6. Cách 2: Dummy + tail cho tầng dưới — `O(1)` bộ nhớ

Đi ngang tầng hiện tại theo `next` (đã nối), và **nối dài** một danh sách liên kết cho tầng dưới:

```text
cur = root
while cur:                         # cur: đầu tầng hiện tại
    dummy = Node(); tail = dummy   # danh sách tầng dưới, bắt đầu rỗng
    node = cur
    while node:                    # đi ngang tầng hiện tại
        if node.left:  tail.next = node.left;  tail = tail.next
        if node.right: tail.next = node.right; tail = tail.next
        node = node.next
    cur = dummy.next               # đầu tầng dưới
```

`dummy` giúp không phải xử lý riêng "nút đầu tiên của tầng dưới". Mỗi tầng chỉ tạo **một** nút giả → bộ nhớ phụ `O(1)` (có thể dùng lại một `dummy` duy nhất).

In [17]:
def connect_o1(root: Optional[Node]) -> Optional[Node]:
    cur = root
    dummy = Node()
    while cur:
        dummy.next = None
        tail = dummy
        node = cur
        while node:
            if node.left:
                tail.next = node.left
                tail = tail.next
            if node.right:
                tail.next = node.right
                tail = tail.next
            node = node.next
        cur = dummy.next
    return root

print(serialize_next(connect_o1(build_tree(EX))))

[1, '#', 2, 3, '#', 4, 5, 7, '#']


Quan trọng: `dummy.next = None` ở đầu mỗi tầng. Nếu tầng dưới rỗng (đã tới lá), `dummy.next` vẫn là `None` → vòng ngoài dừng.

# 7. Mô phỏng

In [18]:
def trace_o1(values):
    root = build_tree(values)
    cur, depth, dummy = root, 0, Node("D")
    while cur:
        dummy.next = None
        tail = dummy
        node = cur
        print(f"Tầng {depth}: đi ngang từ {cur.val}")
        while node:
            for c in (node.left, node.right):
                if c:
                    tail.next = c
                    print(f"   từ {node.val}: nối {tail.val} -> {c.val}")
                    tail = c
            node = node.next
        cur = dummy.next
        depth += 1
    print("Kết quả:", serialize_next(root))
    return root

_ = trace_o1([1, 2, 3, 4, 5, None, 7])
print()
_ = trace_o1([1, 2, 3, 4, None, None, 5, 6, None, None, 7])

Tầng 0: đi ngang từ 1


   từ 1: nối D -> 2
   từ 1: nối 2 -> 3
Tầng 1: đi ngang từ 2
   từ 2: nối D -> 4
   từ 2: nối 4 -> 5
   từ 3: nối 5 -> 7
Tầng 2: đi ngang từ 4
Kết quả: [1, '#', 2, 3, '#', 4, 5, 7, '#']

Tầng 0: đi ngang từ 1
   từ 1: nối D -> 2
   từ 1: nối 2 -> 3
Tầng 1: đi ngang từ 2
   từ 2: nối D -> 4
   từ 3: nối 4 -> 5
Tầng 2: đi ngang từ 4
   từ 4: nối D -> 6
   từ 5: nối 6 -> 7
Tầng 3: đi ngang từ 6
Kết quả: [1, '#', 2, 3, '#', 4, 5, '#', 6, 7, '#']


Dòng `nối D -> x` là lúc gắn nút đầu tiên của tầng dưới vào nút giả — **không** gán `next` cho nút thật nào.

# 8. Phiên bản đúng format LeetCode

In [19]:
class Solution:
    def connect(self, root: 'Node') -> 'Node':
        cur = root
        dummy = Node()
        while cur:
            dummy.next = None
            tail = dummy
            node = cur
            while node:
                if node.left:
                    tail.next = node.left
                    tail = tail.next
                if node.right:
                    tail.next = node.right
                    tail = tail.next
                node = node.next
            cur = dummy.next
        return root

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<12} PASS  {actual}")


def random_tree(n):
    vals = iter(random.sample(range(-100, 101), n))
    def make(k):
        if k == 0:
            return None
        left = random.randint(0, k - 1)
        node = Node(next(vals))
        node.left = make(left)
        node.right = make(k - 1 - left)
        return node
    return make(n)


def next_map(root):
    res, q = {}, deque([root] if root else [])
    while q:
        n = q.popleft()
        res[n.val] = n.next.val if n.next else None
        for c in (n.left, n.right):
            if c:
                q.append(c)
    return res


sol = Solution()
check("Ví dụ 1", serialize_next(sol.connect(build_tree(EX))), [1, "#", 2, 3, "#", 4, 5, 7, "#"])
check("Ví dụ 2", serialize_next(sol.connect(build_tree([]))), [])
check("Lỗ giữa", serialize_next(sol.connect(build_tree([1, 2, 3, 4, None, None, 5]))), [1, "#", 2, 3, "#", 4, 5, "#"])

random.seed(117)
for _ in range(3000):
    n = random.randint(0, 20)
    s = random.random()
    random.seed(s); a = sol.connect(random_tree(n))
    random.seed(s); b = connect_bfs(random_tree(n))
    random.seed(s); ref = random_tree(n)
    assert next_map(a) == next_map(b)
    assert serialize_next(a) == expected_levels(ref)
print("3000 cây ngẫu nhiên: cách O(1) khớp BFS, đọc theo next đúng level order")

Ví dụ 1      PASS  [1, '#', 2, 3, '#', 4, 5, 7, '#']
Ví dụ 2      PASS  []
Lỗ giữa      PASS  [1, '#', 2, 3, '#', 4, 5, '#']


3000 cây ngẫu nhiên: cách O(1) khớp BFS, đọc theo next đúng level order


# 9. Vì sao đúng? Độ phức tạp?

**Bất biến:** khi vòng ngoài bắt đầu với `cur`, tầng chứa `cur` đã được nối đúng bằng `next` (từ trái sang phải), và `cur` là nút đầu tầng.

- Đi ngang tầng theo `next`, với mỗi nút thêm con trái rồi con phải vào cuối danh sách `dummy → …` → danh sách thu được chính là tầng dưới theo đúng thứ tự trái sang phải, các nút liền kề được nối bằng `next`.
- Nút cuối tầng dưới giữ `next = None` (ban đầu là `None`, không bị gán).
- `dummy.next` là đầu tầng dưới → bất biến giữ cho vòng sau.

```text
Time:  O(n)
Space: O(1)  (một nút giả dùng lại)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Không đặt lại `dummy.next` mỗi tầng

In [21]:
def wrong_no_reset(root):
    cur, dummy = root, Node()
    steps = 0
    while cur and steps < 20:
        tail = dummy                               # SAI: thiếu dummy.next = None
        node = cur
        while node:
            for c in (node.left, node.right):
                if c:
                    tail.next = c; tail = c
            node = node.next
        cur = dummy.next
        steps += 1
    return steps

print("Sai : vòng ngoài chạy", wrong_no_reset(build_tree(EX)), "lần (bị chặn ở 20) thay vì 3")

Sai : vòng ngoài chạy 20 lần (bị chặn ở 20) thay vì 3


Ở tầng lá không có con nào, `dummy.next` vẫn trỏ tới đầu tầng **trước** → lặp mãi.

## Lỗi 2: Dùng quy tắc của #116

Như mục 4.

## Lỗi 3: Đệ quy (trái trước) với hàm "tìm next của con"

Một cách đệ quy phổ biến: với mỗi nút, tìm `getNext(node.next)` = con đầu tiên của các nút bên phải. Nhưng nếu đệ quy **trái trước phải**, khi xử lý cây con trái, các `next` ở phía phải **chưa** được nối → `getNext` dừng sớm.

In [22]:
def wrong_recursive_left_first(root):
    def first_child_to_right(node):
        while node:
            if node.left:
                return node.left
            if node.right:
                return node.right
            node = node.next
        return None
    def go(node):
        if not node:
            return
        if node.left:
            node.left.next = node.right or first_child_to_right(node.next)
        if node.right:
            node.right.next = first_child_to_right(node.next)
        go(node.left)                              # SAI: phải đi node.right trước
        go(node.right)
    go(root)
    return root

src = [1, 2, 3, None, 4, 5, 6, 7, 8, None, None, 9]
print("Mong đợi:", expected_levels(build_tree(src)))
print("Sai     :", serialize_next(wrong_recursive_left_first(build_tree(src))))

Mong đợi: [1, '#', 2, 3, '#', 4, 5, 6, '#', 7, 8, 9, '#']
Sai     : [1, '#', 2, 3, '#', 4, 5, 6, '#', 7, 8, '#']


Khi xử lý nút `4` (thuộc nhánh trái), cần tìm con đầu tiên bên phải của `8` bằng cách đi `4.next = 5`, rồi `5.next`... Nhưng `5` không có con, và `5.next = 6` chỉ được gán khi xử lý nút `3` — **sau** nhánh trái. Nên `first_child_to_right` dừng ở `5` và `8.next` không tìm được `9`. Đổi thành đệ quy **phải trước trái** thì đúng — vì khi xử lý cây trái, mọi `next` phía phải đã có (Bài 3).

## Lỗi 4: Quên trả về `root`

LeetCode yêu cầu trả về gốc.

# 11. Mẹo nhận diện pattern

**Xây tầng dưới như danh sách liên kết** (dummy + tail) trong khi **đi ngang tầng trên theo `next`** — kỹ thuật chuẩn cho mọi bài "duyệt theo tầng với `O(1)` bộ nhớ" khi đã có con trỏ ngang.

Dummy + tail cũng là khuôn mẫu chung khi **nối dài** một danh sách liên kết (#2, #21, #86).

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Nối `next` cho cây `[1, 2, 3, None, 4, None, 5, 6]`. `4.next` là gì? `6.next`?

## Bài 2 — Điền chỗ trống

In [23]:
def exercise_2_connect(root: Optional[Node]) -> Optional[Node]:
    cur, dummy = root, Node()
    while cur:
        dummy.next = None
        tail = dummy
        node = cur
        while node:
            # TODO: nối con trái / phải (nếu có) vào tail; đi ngang
            pass
        cur = dummy.next
    return root


# assert serialize_next(exercise_2_connect(build_tree([1, 2, 3, 4, 5, None, 7]))) == [1, "#", 2, 3, "#", 4, 5, 7, "#"]
# assert serialize_next(exercise_2_connect(build_tree([]))) == []

## Bài 3 — Biến thể: đệ quy phải trước trái

Sửa `wrong_recursive_left_first` cho đúng bằng cách đệ quy **phải trước**.

In [24]:
def exercise_3_connect_recursive(root: Optional[Node]) -> Optional[Node]:
    # TODO
    pass


# assert serialize_next(exercise_3_connect_recursive(build_tree([1, 2, 3, 4, 5, None, 6, 7, None, None, None, None, 8]))) == [1, "#", 2, 3, "#", 4, 5, 6, "#", 7, 8, "#"]

## Bài 4 — Biến thể: Binary Tree Right Side View (LeetCode #199) với `O(1)` bộ nhớ phụ

Sau khi `connect`, nút nhìn thấy từ bên phải ở mỗi tầng là nút **cuối** khi đi theo `next`. Trả danh sách đó.

```python
right_view([1, 2, 3, None, 5, None, 4]) -> [1, 3, 4]
```

In [25]:
def exercise_4_right_view(root: Optional[Node]) -> List[int]:
    # TODO
    pass


# assert exercise_4_right_view(build_tree([1, 2, 3, None, 5, None, 4])) == [1, 3, 4]
# assert exercise_4_right_view(build_tree([1, 2, 3, 4])) == [1, 3, 4]
# assert exercise_4_right_view(build_tree([])) == []

## Bài 5 — Mini challenge: số nút mỗi tầng chỉ bằng `next`

Sau khi `connect`, trả về độ rộng (số nút) của mỗi tầng, chỉ dùng `next` và một biến "đầu tầng sau".

In [26]:
def exercise_5_level_sizes(root: Optional[Node]) -> List[int]:
    # TODO
    pass


# assert exercise_5_level_sizes(Solution().connect(build_tree([1, 2, 3, 4, 5, None, 7]))) == [1, 2, 3]
# assert exercise_5_level_sizes(None) == []

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_o1` sau khi tự làm.

**Bài 3**
- Mức 1: giữ nguyên phần gán `next` cho hai con, chỉ đổi thứ tự: `go(node.right)` trước `go(node.left)`.

**Bài 4**
- Mức 1: `connect` trước; rồi `head = root`; đi theo `next` tới cuối tầng; đầu tầng sau = con đầu tiên tìm được.

**Bài 5**
- Mức 1: như `serialize_next`, nhưng đếm thay vì ghi giá trị.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [27]:
nm = next_map(sol.connect(build_tree([1, 2, 3, None, 4, None, 5, 6])))
print("4 ->", nm[4], "| 6 ->", nm[6])
_ = trace_o1([1, 2, 3, None, 4, None, 5, 6])

4 -> 5 | 6 -> None
Tầng 0: đi ngang từ 1
   từ 1: nối D -> 2
   từ 1: nối 2 -> 3
Tầng 1: đi ngang từ 2
   từ 2: nối D -> 4
   từ 3: nối 4 -> 5
Tầng 2: đi ngang từ 4
   từ 4: nối D -> 6
Tầng 3: đi ngang từ 6
Kết quả: [1, '#', 2, 3, '#', 4, 5, '#', 6, '#']


## Đáp án Bài 2

In [28]:
def answer_2_connect(root: Optional[Node]) -> Optional[Node]:
    return connect_o1(root)

assert serialize_next(answer_2_connect(build_tree([1, 2, 3, 4, 5, None, 7]))) == [1, "#", 2, 3, "#", 4, 5, 7, "#"]
assert serialize_next(answer_2_connect(build_tree([]))) == []
print("Bài 2 PASS (giống connect_o1 ở mục 6)")

Bài 2 PASS (giống connect_o1 ở mục 6)


## Đáp án Bài 3

In [29]:
def answer_3_connect_recursive(root: Optional[Node]) -> Optional[Node]:
    def first_child_to_right(node):
        while node:
            if node.left:
                return node.left
            if node.right:
                return node.right
            node = node.next
        return None
    def go(node):
        if not node:
            return
        if node.left:
            node.left.next = node.right or first_child_to_right(node.next)
        if node.right:
            node.right.next = first_child_to_right(node.next)
        go(node.right)
        go(node.left)
    go(root)
    return root

assert serialize_next(answer_3_connect_recursive(build_tree([1, 2, 3, 4, 5, None, 6, 7, None, None, None, None, 8]))) == [1, "#", 2, 3, "#", 4, 5, 6, "#", 7, 8, "#"]
for _ in range(2000):
    n = random.randint(0, 20); s = random.random()
    random.seed(s); a = answer_3_connect_recursive(random_tree(n))
    random.seed(s); b = connect_bfs(random_tree(n))
    assert next_map(a) == next_map(b)
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [30]:
def answer_4_right_view(root: Optional[Node]) -> List[int]:
    connect_o1(root)
    out, head = [], root
    while head:
        node, nxt_head = head, None
        while True:
            if nxt_head is None:
                nxt_head = node.left or node.right
            if not node.next:
                break
            node = node.next
        out.append(node.val)
        head = nxt_head
    return out

assert answer_4_right_view(build_tree([1, 2, 3, None, 5, None, 4])) == [1, 3, 4]
assert answer_4_right_view(build_tree([1, 2, 3, 4])) == [1, 3, 4]
assert answer_4_right_view(build_tree([])) == []
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [31]:
def answer_5_level_sizes(root: Optional[Node]) -> List[int]:
    out, head = [], root
    while head:
        cnt, node, nxt_head = 0, head, None
        while node:
            cnt += 1
            if nxt_head is None:
                nxt_head = node.left or node.right
            node = node.next
        out.append(cnt)
        head = nxt_head
    return out

assert answer_5_level_sizes(Solution().connect(build_tree([1, 2, 3, 4, 5, None, 7]))) == [1, 2, 3]
assert answer_5_level_sizes(None) == []
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> Đi ngang tầng hiện tại theo `next`; với mỗi con (trái rồi phải), nối vào `tail` của một danh sách bắt đầu bằng `dummy`. Xong tầng thì `cur = dummy.next`. Nhớ đặt lại `dummy.next = None` mỗi tầng. `O(n)` thời gian, `O(1)` bộ nhớ.

---

## Checklist trước khi nộp

- [ ] Không giả định nút có đủ hai con?
- [ ] `dummy.next = None` đầu mỗi tầng?
- [ ] Thêm con trái trước con phải?
- [ ] Nếu đệ quy: đi **phải trước trái**?